# Prompt Engineering
In this notebook, we will explore the concept of prompt engineering.

The goal is to move from a single hand-written prompt to a reusable prompt template and then to a few-shot workflow that gives the model clearer formatting guidance.

By the end, you should be able to:
- explain the four common prompt building blocks
- convert a one-off prompt into a reusable `PromptTemplate`
- add an output parser so the chain returns plain text
- compare zero-shot and few-shot prompting for formatting-heavy tasks

## Lesson Map
This notebook progresses through three increasingly reusable prompt patterns.

```mermaid
flowchart LR
    A[One-off prompt] --> B[PromptTemplate]
    B --> C[PromptTemplate + StrOutputParser]
    C --> D[Few-shot prompt]
    D --> E[FewShotPromptTemplate]
```

At each step, the prompt becomes easier to reuse, debug, and adapt to new inputs.

## Structure of a Prompt
A prompt is a structured input that guides the model's response. It typically consists of:
- **Instruction**: A clear directive on what the model should do. Typically how it should use inputs and or external information to produce the desired output.
- **External information or context**: Additional data that we can manually provide the model prompt with, retrieve from a vector database, or retrieve from a web search.
- **User input**: the specific query that the user inputs.
- **Output indicator**: the beginning of the model's response, which can be a specific phrase or format that the model should follow. For example, "The answer is:" or "Response:"

```mermaid
flowchart TD
    A[Instruction] --> E[Final prompt]
    B[Context] --> E
    C[User input] --> E
    D[Output indicator] --> E
    E --> F[Model response]
```

A good prompt does not need to be long. It needs to be clear about what the model should use, what it should produce, and what information changes from request to request.

Not all prompts will have all these components, but a well-structured prompt will typically include at least two or more of them. In practice, adding explicit context and a visible output cue usually makes the model's behavior easier to control.

In [1]:
# Core imports for loading credentials, calling the Groq model, and templating prompts.
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate


# Normalize LangChain message content before printing so examples stay type-safe.
def render_message_content(content: object) -> str:
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "".join(part if isinstance(part, str) else str(part) for part in content)
    return str(content)

In [2]:
load_dotenv()

True

In [3]:
# Start with one fully written prompt so we can inspect the structure directly.
# After that, we will convert it into a reusable template with a variable slot.
prompt = """Answer the question based on the context below. If the
question cannot be answered using the information provided answer
with "I don't know".

Context: Geoffrey Everest Hinton (born 6 December 1947) is a British-Canadian computer scientist, cognitive scientist, 
cognitive psychologist, known for his work on artificial neural networks which earned him the title as the 
"Godfather of AI". Hinton is University Professor Emeritus at the University of Toronto. From 2013 to 2023, 
he divided his time working for Google (Google Brain) and the University of Toronto, before publicly announcing 
his departure from Google in May 2023, citing concerns about the risks of artificial intelligence (AI) technology.

In 2017, he co-founded and became the chief scientific advisor of the Vector Institute in Toronto.
With David Rumelhart and Ronald J. Williams, Hinton was co-author of a highly cited paper published in 1986 
that popularised the backpropagation algorithm for training multi-layer neural networks, although they were 
not the first to propose the approach. Hinton is viewed as a leading figure in the deep learning community.
The image-recognition milestone of the AlexNet designed in collaboration with his students Alex Krizhevsky 
and Ilya Sutskever for the ImageNet challenge 2012[22] was a breakthrough in the field of computer vision.

Hinton received the 2018 Turing Award, often referred to as the "Nobel Prize of Computing", together with 
Yoshua Bengio and Yann LeCun, for their work on deep learning. They are sometimes referred to as the 
"Godfathers of Deep Learning", and have continued to give public talks together. He was also awarded 
the 2024 Nobel Prize in Physics, shared with John Hopfield.

Question: Which important achievements did he get?

Answer: """

In this example, we have:
* **Instruction**: "Answer the question based on the provided context."
* **External information or context**: The provided text about Hinton.
* **User input**: "Which important achievements did he get?"
* **Output indicator**: "Answer:"

This structure matters because it separates stable guidance from changing inputs. The instruction and context stay mostly fixed, while the user question can change from one run to the next.

In [4]:
# Keep the model configuration lightweight for fast, deterministic demos.
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.1,
)

In [5]:
# Send the fully written prompt directly to the model.
answer = llm.invoke(prompt)

# Print the text content of the model response.
print(render_message_content(answer.content).strip())

He is best known for the following major achievements:

1. **Co‑authoring the 1986 paper that popularised the back‑propagation algorithm** for training multi‑layer neural networks (with David Rumelhart and Ronald J. Williams).  
2. **Co‑creating AlexNet** with Alex Krizhevsky and Ilya Sutskever, which won the 2012 ImageNet challenge and sparked the modern deep‑learning revolution in computer vision.  
3. **Co‑founding the Vector Institute in Toronto** in 2017 and serving as its chief scientific advisor.  
4. **Receiving the 2018 Turing Award** (often called the “Nobel Prize of Computing”) jointly with Yoshua Bengio and Yann LeCun for their foundational work on deep learning.  
5. **Being awarded the 2024 Nobel Prize in Physics** (shared with John Hopfield) for contributions that have had a profound impact on the field.


We would not typically know the user input in advance. So instead of writing the full prompt every time, we can use a template to create it dynamically.

This is one of the most practical prompt-engineering moves in an application: keep the stable instructions in one place and inject only the part that changes.

In [6]:
# Reuse the same prompt structure, but replace the question with a template variable.
prompt = """Answer the question based on the context below. If the
question cannot be answered using the information provided answer
with "I don't know".

Context: Geoffrey Everest Hinton (born 6 December 1947) is a British-Canadian computer scientist, cognitive scientist, 
cognitive psychologist, known for his work on artificial neural networks which earned him the title as the 
"Godfather of AI". Hinton is University Professor Emeritus at the University of Toronto. From 2013 to 2023, 
he divided his time working for Google (Google Brain) and the University of Toronto, before publicly announcing 
his departure from Google in May 2023, citing concerns about the risks of artificial intelligence (AI) technology.

In 2017, he co-founded and became the chief scientific advisor of the Vector Institute in Toronto.
With David Rumelhart and Ronald J. Williams, Hinton was co-author of a highly cited paper published in 1986 
that popularised the backpropagation algorithm for training multi-layer neural networks, although they were 
not the first to propose the approach. Hinton is viewed as a leading figure in the deep learning community.
The image-recognition milestone of the AlexNet designed in collaboration with his students Alex Krizhevsky 
and Ilya Sutskever for the ImageNet challenge 2012[22] was a breakthrough in the field of computer vision.

Hinton received the 2018 Turing Award, often referred to as the "Nobel Prize of Computing", together with 
Yoshua Bengio and Yann LeCun, for their work on deep learning. They are sometimes referred to as the 
"Godfathers of Deep Learning", and have continued to give public talks together. He was also awarded 
the 2024 Nobel Prize in Physics, shared with John Hopfield.

Question: {query}

Answer: """

# PromptTemplate turns the string into a reusable object with one required input variable.
prompt_template = PromptTemplate(
    input_variables=["query"],
    template=prompt,
)

In [7]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

The `output_parser` converts the model response into a plain Python string. That is helpful when you want the rest of your application to work with text instead of a richer message object.

Without it, many chat-model integrations return a message object that still contains metadata. That richer object is useful in some applications, but plain text is easier for a first prompt-engineering workflow.

In [8]:
# Compose the prompt template, model, and output parser into one reusable pipeline.
chain = prompt_template | llm | output_parser

# Now only the query changes between runs.
answer = chain.invoke({"query": "Which important achievements did he get?"})
print(answer)

He is credited with several landmark achievements in artificial intelligence and machine learning, including:

1. **Popularizing the back‑propagation algorithm** – co‑authoring the 1986 paper with David Rumelhart and Ronald J. Williams that made back‑propagation a practical training method for multi‑layer neural networks.  
2. **Leading the creation of AlexNet** – guiding his students Alex Krizhevsky and Ilya Sutskever to develop the AlexNet architecture that won the 2012 ImageNet challenge and sparked the deep‑learning revolution in computer vision.  
3. **Co‑founding the Vector Institute** – establishing the Toronto‑based research institute in 2017 and serving as its chief scientific advisor.  
4. **Receiving the 2018 Turing Award** – jointly awarded with Yoshua Bengio and Yann LeCun for foundational work on deep learning, often called the “Nobel Prize of Computing.”  
5. **Being awarded the 2024 Nobel Prize in Physics** – shared with John Hopfield for contributions that bridged phys

#### Inspect the Rendered Template
Before calling the model, it is often helpful to inspect the exact prompt string created by `PromptTemplate`. This is especially useful when a prompt behaves unexpectedly, because many prompt issues are really formatting issues.

In [9]:
# Preview the final prompt text for a different user query.
print(prompt_template.format(query="What concerns did Hinton express about AI?"))

Answer the question based on the context below. If the
question cannot be answered using the information provided answer
with "I don't know".

Context: Geoffrey Everest Hinton (born 6 December 1947) is a British-Canadian computer scientist, cognitive scientist, 
cognitive psychologist, known for his work on artificial neural networks which earned him the title as the 
"Godfather of AI". Hinton is University Professor Emeritus at the University of Toronto. From 2013 to 2023, 
he divided his time working for Google (Google Brain) and the University of Toronto, before publicly announcing 
his departure from Google in May 2023, citing concerns about the risks of artificial intelligence (AI) technology.

In 2017, he co-founded and became the chief scientific advisor of the Vector Institute in Toronto.
With David Rumelhart and Ronald J. Williams, Hinton was co-author of a highly cited paper published in 1986 
that popularised the backpropagation algorithm for training multi-layer neural netw

#### Reuse the Same Chain for Multiple Questions
Once the prompt structure is stable, reusing it becomes easy. The same chain can answer several related questions as long as each question matches the expected input schema.

In [10]:
# Reuse the same prompt template across a small set of related queries.
queries = [
    "Which important achievements did he get?",
    "What role did AlexNet play in computer vision?",
    "What AI risks did Hinton mention?",
]

for query in queries:
    print("=" * 100)
    print(f"Query: {query}")
    print(chain.invoke({"query": query}))

Query: Which important achievements did he get?
Answer:  
- Co‑author of the 1986 paper that popularised the back‑propagation algorithm for training multi‑layer neural networks.  
- Co‑founder and chief scientific advisor of the Vector Institute in Toronto (2017).  
- Key contributor to the AlexNet image‑recognition system that won the ImageNet challenge in 2012.  
- Recipient of the 2018 Turing Award (along with Yoshua Bengio and Yann LeCun) for pioneering work on deep learning.  
- Awarded the 2024 Nobel Prize in Physics (shared with John Hopfield).
Query: What role did AlexNet play in computer vision?
AlexNet was a breakthrough milestone in computer vision, demonstrating that deep convolutional neural networks could dramatically outperform previous methods on large‑scale image‑recognition tasks and sparking widespread adoption of deep learning in the field.
Query: What AI risks did Hinton mention?
I don't know.


## Few-Shot Prompting
Few-shot prompting means we include a small number of examples in the prompt so the model can imitate the structure, tone, or output format we want.

Two useful knowledge sources to keep in mind are:
+ **Parametric knowledge**: information the model learned during training and stores in its parameters.
+ **Source knowledge**: information we provide at inference time through the prompt, retrieved documents, or tool outputs.

Few-shot examples belong to the second category. They do not retrain the model, but they do give it a stronger pattern to follow for the current request.

```mermaid
flowchart LR
    A[Built-in model knowledge] --> D[Model reasoning]
    B[Task instruction] --> D
    C[Worked examples in prompt] --> D
    D --> E[More controlled output format]
```


### First Try Without Examples
We will begin by asking the model to create an FAQ without showing it an example of the format we want. This gives us a baseline to compare against.

This is still prompt engineering, but it is closer to a zero-shot setup: we describe the task without demonstrating the exact shape of the desired answer.

In [11]:
from IPython.display import Markdown

# This prompt describes the task, but it does not demonstrate the exact format yet.
template = """Create a FAQ in Markdown format from the following questions and answers.
    Q1: Do you ship internationally?,
    A1: Yes, we ship worldwide. Shipping times and costs vary depending on the destination.
    Q2: How can I track my order?,
    A2: Once your order ships, you’ll receive a tracking number via email. You can also track orders through your account dashboard.
    Q3: What is your return policy?,
    A3: You can return items within 30 days of receipt for a full refund. Items must be in original condition and packaging.

    """


prompt_template = PromptTemplate(
    template=template,
    input_variables=[],  # This first example is fixed, so it has no dynamic inputs.
)

chain = prompt_template | llm | output_parser

answer = chain.invoke({})
Markdown(answer)

# Frequently Asked Questions

| Question | Answer |
|----------|--------|
| **Do you ship internationally?** | Yes, we ship worldwide. Shipping times and costs vary depending on the destination. |
| **How can I track my order?** | Once your order ships, you’ll receive a tracking number via email. You can also track orders through your account dashboard. |
| **What is your return policy?** | You can return items within 30 days of receipt for a full refund. Items must be in original condition and packaging. |

--- 

**Quick Tips**

- **International Shipping** – Check the shipping calculator on our site for estimated delivery times and fees.
- **Tracking** – Look for the “Track Order” link in the confirmation email or log in to your account to view the status.
- **Returns** – Keep the original packaging and any tags; this speeds up the refund process.

The output may be acceptable, but it can still drift in structure. If we want a very specific layout, we can show the model an example of the desired format directly in the prompt. That is the core idea behind few-shot prompting.

In [12]:
# Here we include one complete example input-output pair inside the prompt.
template = """
Create a structured Markdown FAQ with links, headers, and bullet points based on the following questions and answers.

**Example:**

Example Input:
Q1: Do you ship internationally?
A1: Yes, we ship worldwide. Shipping times and costs vary depending on the destination.
Q2: How can I track my order?
A2: Once your order ships, you’ll receive a tracking number via email. You can also track orders through your account dashboard.
Q3: What is your return policy?
A3: You can return items within 30 days of receipt for a full refund. Items must be in original condition and packaging.

Example Output:
# Frequently Asked Questions
## Shipping
- **Do you ship internationally?**  
  Yes, we ship worldwide. Shipping times and costs vary depending on the destination.
- **How can I track my order?**  
  Once your order ships, you’ll receive a tracking number via email. You can also track orders through your account dashboard.
## Returns
- **What is your return policy?**  
  You can return items within 30 days of receipt for a full refund. Items must be in original condition and packaging.

Now, create a similar FAQ based on the following questions and answers:
Q1: What payment methods are accepted?,
A1: We accept Visa, MasterCard, PayPal, Apple Pay, and Klarna (Buy Now, Pay Later in select countries).,
Q2: Are your products eco-friendly?,
A2: Yes, we use sustainable materials and practices in our production process.,
Q3: Can I change or cancel my order after placing it?,
A3: We can modify or cancel orders within 1 hour of purchase. Contact our support team immediately for assistance.
"""

In [13]:
prompt_template = PromptTemplate(
    template=template,
    input_variables=[],  # This prompt is still fixed because the target FAQ is embedded in the string.
)

chain = prompt_template | llm | output_parser

answer = chain.invoke({})
Markdown(answer)

# Frequently Asked Questions

## Payments  
- **What payment methods are accepted?**  
  We accept Visa, MasterCard, PayPal, Apple Pay, and Klarna (Buy Now, Pay Later in select countries).  
  👉 For a full list of supported cards and payment options, see our [Payment Methods](https://example.com/payment-methods) page.

## Sustainability  
- **Are your products eco‑friendly?**  
  Yes, we use sustainable materials and practices throughout our production process.  
  🌱 Learn more about our environmental initiatives on the [Sustainability](https://example.com/sustainability) page.

## Orders  
- **Can I change or cancel my order after placing it?**  
  We can modify or cancel orders within **1 hour** of purchase. Contact our support team immediately for assistance.  
  📄 For step‑by‑step instructions, visit the [Order Management](https://example.com/orders) guide.

We got a more readable output by adding a few-shot example to the prompt.

In LangChain, we can also use `FewShotPromptTemplate` to build the same idea in a cleaner, more reusable way.

```mermaid
flowchart LR
    A[Stored example pairs] --> B[FewShotPromptTemplate]
    C[New user input] --> B
    B --> D[Formatted prompt]
    D --> E[Model output]
```


To build a reusable few-shot prompt, we break the prompt into smaller parts:
- a list of example input-output pairs
- a template for rendering each example
- a `prefix` with the high-level instruction
- a `suffix` where the new user input is inserted

This decomposition is useful because you can swap examples, update the instruction, or change the user input without rewriting the entire prompt string.

In [14]:
from langchain_core.prompts import FewShotPromptTemplate

In [15]:
# Each example teaches the model one slice of the desired FAQ formatting style.
examples = [
    {
        "input": (
            "Q1: Do you ship internationally?\n"
            "A1: Yes, we ship worldwide. Shipping times and costs vary depending on the destination.\n"
        ),
        "output": (
            "## Shipping\n"
            "- **Do you ship internationally?**  \n"
            "  Yes, we ship worldwide. Shipping times and costs vary depending on the destination.\n"
        ),
    },
    {
        "input": (
            "Q2: How can I track my order?\n"
            "A2: Once your order ships, you’ll receive a tracking number via email. You can also track orders through your account dashboard.\n"
        ),
        "output": (
            "## Tracking\n"
            "- **How can I track my order?**  \n"
            "  Once your order ships, you’ll receive a tracking number via email. You can also track orders through your account dashboard.\n"
        ),
    },
    {
        "input": (
            "Q3: What is your return policy?\n"
            "A3: You can return items within 30 days of receipt for a full refund. Items must be in original condition and packaging.\n"
        ),
        "output": (
            "## Returns\n"
            "- **What is your return policy?**  \n"
            "  You can return items within 30 days of receipt for a full refund. Items must be in original condition and packaging.\n"
        ),
    },
]

In [16]:
# Define how one example pair should be rendered inside the final prompt.
example_template = """Example input:
{input}

Example Output:
# Frequently Asked Questions\n\n
{output}
"""

In [17]:
# Wrap the single-example template in a PromptTemplate so LangChain can fill it repeatedly.
example_prompt = PromptTemplate(
    input_variables=["input", "output"], template=example_template
)

In [18]:
# The prefix explains the overall task before any examples are shown.
prefix = """Create a structured Markdown FAQ with links, headers, and bullet points based on the following questions and answers.

**Example:**"""

# The suffix tells LangChain where the new user input should be inserted.
suffix = """
Now, create a similar FAQ based on the following questions and answers:
{input}"""

In [19]:
# Combine examples, prefix, suffix, and the example renderer into one reusable prompt object.
few_shot = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix=prefix,
    suffix=suffix,
    input_variables=["input"],
    example_separator="\n",
)

In [20]:
# This is the new data we want to transform using the same pattern as the examples.
user_input = """Q1: What payment methods are accepted?,
A1: We accept Visa, MasterCard, PayPal, Apple Pay, and Klarna (Buy Now, Pay Later in select countries).,
Q2: Are your products eco-friendly?,
A2: Yes, we use sustainable materials and practices in our production process.,
Q3: Can I change or cancel my order after placing it?,
A3: We can modify or cancel orders within 1 hour of purchase. Contact our support team immediately for assistance.
"""

Now let's inspect the final prompt text that LangChain creates before it is sent to the model. Looking at the formatted prompt is a good debugging habit when a prompt is not producing the structure you expect.

When a prompt fails, this is often the fastest place to debug: check whether the examples, separators, and user input were assembled the way you intended.

In [21]:
# Render the full few-shot prompt as plain text so we can inspect it before inference.
formatted_prompt = few_shot.format(input=user_input)
print(formatted_prompt)

Create a structured Markdown FAQ with links, headers, and bullet points based on the following questions and answers.

**Example:**
Example input:
Q1: Do you ship internationally?
A1: Yes, we ship worldwide. Shipping times and costs vary depending on the destination.


Example Output:
# Frequently Asked Questions


## Shipping
- **Do you ship internationally?**  
  Yes, we ship worldwide. Shipping times and costs vary depending on the destination.


Example input:
Q2: How can I track my order?
A2: Once your order ships, you’ll receive a tracking number via email. You can also track orders through your account dashboard.


Example Output:
# Frequently Asked Questions


## Tracking
- **How can I track my order?**  
  Once your order ships, you’ll receive a tracking number via email. You can also track orders through your account dashboard.


Example input:
Q3: What is your return policy?
A3: You can return items within 30 days of receipt for a full refund. Items must be in original condi

After we verify the formatted prompt, we can send that reusable few-shot pipeline to the model.

This final step is the most production-like one in the notebook: the examples stay fixed, the user input changes, and LangChain handles the assembly for us.

In [22]:
# Reuse the few-shot prompt pipeline on the new FAQ data.
chain = few_shot | llm | output_parser

answer = chain.invoke({"input": user_input})
Markdown(answer)

# Frequently Asked Questions

## Payment Methods  
- **What payment methods are accepted?**  
  We accept Visa, MasterCard, PayPal, Apple Pay, and Klarna (Buy Now, Pay Later in select countries).  
  *Learn more about our payment options →* [Payment Methods](https://example.com/payment-methods)

## Sustainability  
- **Are your products eco‑friendly?**  
  Yes, we use sustainable materials and practices in our production process.  
  *Discover our environmental commitments →* [Sustainability](https://example.com/sustainability)

## Order Management  
- **Can I change or cancel my order after placing it?**  
  We can modify or cancel orders within 1 hour of purchase. Contact our support team immediately for assistance.  
  *Read our order‑change policy →* [Order Management](https://example.com/order-management)

#### Reuse the Few-Shot Prompt on New Data
The real strength of `FewShotPromptTemplate` is that the example set stays fixed while the incoming data changes. That makes it easy to test whether your examples generalize beyond the first input.

In [23]:
# Try the same few-shot prompt on a second FAQ dataset.
second_user_input = """Q1: Do you offer gift cards?,
A1: Yes, digital gift cards are available in several amounts and are delivered by email.,
Q2: How long does delivery take?,
A2: Standard delivery usually takes 3 to 5 business days, while express delivery takes 1 to 2 business days.,
Q3: What should I do if my package is damaged?,
A3: Contact support within 48 hours and include photos of the package and product so we can help quickly.
"""

Markdown(chain.invoke({"input": second_user_input}))

# Frequently Asked Questions

## Gift Cards
- **Do you offer gift cards?**  
  Yes, digital gift cards are available in several amounts and are delivered by email.  
  For more details, visit our [Gift Card page](#gift-cards).

## Delivery
- **How long does delivery take?**  
  Standard delivery usually takes 3 to 5 business days, while express delivery takes 1 to 2 business days.  
  Check our [Shipping & Delivery](#delivery) page for more information.

## Damaged Packages
- **What should I do if my package is damaged?**  
  Contact support within 48 hours and include photos of the package and product so we can help quickly.  
  Reach out via our [Support Center](#support).

## Things to Test Next
If you want to keep exploring prompt engineering, here are some practical next steps:
- remove the example output and compare the quality against the few-shot version
- add a stricter output requirement such as JSON, YAML, or a markdown table
- change the examples and observe how the output style shifts
- test how the prompt behaves when the source information is missing or ambiguous
- try multiple user inputs that differ in topic but still need the same output structure

## Conclusion
In this notebook, we have learned:
- how prompt structure influences model behavior
- how to separate stable instructions from dynamic user input with `PromptTemplate`
- how output parsers simplify downstream handling of model responses
- how few-shot examples help the model imitate a desired output format
- how `FewShotPromptTemplate` makes example-driven prompting easier to reuse and debug